In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene
import statsmodels.stats.multitest as smm

import sys
sys.path.append('../../../')
sys.path.append('../../')
from utilis import *
from settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

# Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
print(df.shape) 
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)
print(df.shape) 

## Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
# removing samples with moca below the minimum threshold of 19
df = df[df[MOCA_COL] > MIN_MOCA_SCORE]
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

# Filter for Females

In [ ]:
df_high_vs_low = df_high_vs_low[df_high_vs_low['Gender'] == 'f']

In [ ]:
(df_high_vs_low['p-tau217'] > HIGH_TAU_217_THR).value_counts()

## behavioral tests

In [ ]:
BEHAVIORAL_SAVE_DIR = SAVE_DIR + 'behavioral/'

### TMT (Trail B)

In [ ]:
df_high_vs_low_tmt = df_high_vs_low.dropna(subset=[TMT_B_COL])
df_high_vs_low_tmt[TMT_B_COL] = df_high_vs_low_tmt[TMT_B_COL].astype(float)
graph_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_tmt, TMT_B_COL, TAU_CAT, graph_colors, 'Trail B vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Trail_B_vs_Tau_217_category_kde.pdf')

In [ ]:
sns.violinplot(data=df_high_vs_low, x=TAU_CAT, y=TMT_B_COL, palette=graph_colors, inner='quartile')
plt.title('Trail B vs Tau 217 category')

In [ ]:
# Statistical test
data_type = 'Continuous'
group1 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'Low'][TMT_B_COL]
group2 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'High'][TMT_B_COL]

p_trail_b = stat_test(data_type, group1, group2)

### MBI-C

In [ ]:
df_high_vs_low_mbi = df_high_vs_low.dropna(subset=[MBI_C_COL])
df_high_vs_low_mbi[MBI_C_COL] = df_high_vs_low_mbi[MBI_C_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_mbi, MBI_C_COL, TAU_CAT, graph_colors, 'MBI-C vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'MBI_C_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
sns.violinplot(data=df_high_vs_low_mbi, x=TAU_CAT, y=MBI_C_COL, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('MBI-C vs Tau 217 category')

plt.savefig(SAVE_DIR + 'MBI_C_vs_Tau_217_category_violin.pdf')

In [ ]:
# threshold MBI-C
MBI_C_THR = 6
df_high_vs_low_mbi['MBI-C_THR'] = (df_high_vs_low_mbi[MBI_C_COL] > MBI_C_THR)
# df_high_vs_low_mbi_c_thr['MBI-C_THR']

df_high_vs_low_mbi_c_thr = df_high_vs_low_mbi.dropna(subset=['MBI-C_THR'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_mbi_c_thr, 'MBI-C_THR', TAU_CAT, 2 , graph_colors, 'MBI-C_THR vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Anxiety_Depression_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Ordinal'

group1 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'Low'][MBI_C_COL]
group2 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'High'][MBI_C_COL]

p_mbi_c = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_mbi
group2 = [TAU_CAT, 'MBI-C_THR']

p_mbi_c_thr = stat_test(data_type, group1, group2) # Too few samples for the test

### IST

In [ ]:
df_high_vs_low_ist = df_high_vs_low.dropna(subset=[IST_COL])
df_high_vs_low_ist[IST_COL] = df_high_vs_low_ist[IST_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_ist, IST_COL, TAU_CAT, graph_colors, 'IST vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'IST_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
sns.violinplot(data=df_high_vs_low_ist, x=TAU_CAT, y=IST_COL, hue=TAU_CAT, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('IST vs Tau 217 category')

plt.savefig(BEHAVIORAL_SAVE_DIR + 'IST_vs_Tau_217_category_violin.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'Low'][IST_COL]
group2 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'High'][IST_COL]

p_ist = stat_test(data_type, group1, group2)

### interest, motivation, and drive

In [ ]:
IMD = 'interest, motivation, and drive'

df_high_vs_low_imd = df_high_vs_low.dropna(subset=[IMD])
df_high_vs_low_imd[IMD] = df_high_vs_low_imd[IMD].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_imd, IMD, TAU_CAT, graph_colors, 'Interest, Motivation, and Drive vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Interest_Motivation_and_Drive_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# categorical data
IMD_THR = 2

df_high_vs_low_imd['IMD_THR'] = (df_high_vs_low_imd[IMD] > IMD_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_imd, 'IMD_THR', TAU_CAT, 2 , graph_colors, 'Interest, Motivation, and Drive vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Interest_Motivation_and_Drive_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_imd[df_high_vs_low_imd[TAU_CAT] == 'Low'][IMD]
group2 = df_high_vs_low_imd[df_high_vs_low_imd[TAU_CAT] == 'High'][IMD]

p_intreset = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_imd
group2 = [TAU_CAT, 'IMD_THR']

p_intreset_thr = stat_test(data_type, group1, group2) # Too few samples for the test

### Impulse control 

In [ ]:
IC = 'Impulse control'

df_high_vs_low_ic = df_high_vs_low.dropna(subset=[IC])
df_high_vs_low_ic[IC] = df_high_vs_low_ic[IC].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ic, IC, TAU_CAT, graph_colors, 'Impulse Control vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Impulse_Control_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Impulse control threshold
IC_THR = 2
df_high_vs_low_ic['IC_THR'] = (df_high_vs_low_ic[IC] > IC_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ic, 'IC_THR', TAU_CAT, 2 , graph_colors, 'Impulse Control vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Impulse_Control_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ic[df_high_vs_low_ic[TAU_CAT] == 'Low'][IC]
group2 = df_high_vs_low_ic[df_high_vs_low_ic[TAU_CAT] == 'High'][IC]

p_impulse_control = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_ic
group2 = [TAU_CAT, 'IC_THR']

p_impulse_control_thr = stat_test(data_type, group1, group2) # Too few samples for the test

### following societal norms

In [ ]:
SOCIETAL_NORMS = 'following societal norms'

df_high_vs_low_fcn = df_high_vs_low.dropna(subset=[SOCIETAL_NORMS])
df_high_vs_low_fcn[SOCIETAL_NORMS] = df_high_vs_low_fcn[SOCIETAL_NORMS].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_fcn, SOCIETAL_NORMS, TAU_CAT, graph_colors, 'following societal norms',BEHAVIORAL_SAVE_DIR + 'following societal norms_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# categorical data
FCN_THR = 0
df_high_vs_low_fcn['FCN_THR'] = (df_high_vs_low_fcn[SOCIETAL_NORMS] > FCN_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_fcn, 'FCN_THR', TAU_CAT, 2 , graph_colors, 'following societal norms', BEHAVIORAL_SAVE_DIR + 'following societal norms_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_fcn[df_high_vs_low_fcn[TAU_CAT] == 'Low'][SOCIETAL_NORMS]
group2 = df_high_vs_low_fcn[df_high_vs_low_fcn[TAU_CAT] == 'High'][SOCIETAL_NORMS]

p_following_societal_norms = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_fcn
group2 = [TAU_CAT, 'FCN_THR']

p_following_societal_norms_thr = stat_test(data_type, group1, group2) # Too few samples for the test

### strongly held beliefs and sensory

In [ ]:
STRONGLY_HELD_BELIEFS = 'strongly held beliefs and sensory'

df_high_vs_low_shb = df_high_vs_low.dropna(subset=[STRONGLY_HELD_BELIEFS])
df_high_vs_low_shb[STRONGLY_HELD_BELIEFS] = df_high_vs_low_shb[STRONGLY_HELD_BELIEFS].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_shb, STRONGLY_HELD_BELIEFS, TAU_CAT, graph_colors, 'strongly held beliefs and sensory',BEHAVIORAL_SAVE_DIR + 'strongly held beliefs and sensory_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Categorical data
SHB_THR = 0
df_high_vs_low_shb['SHB_THR'] = (df_high_vs_low_shb[STRONGLY_HELD_BELIEFS] > SHB_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_shb, 'SHB_THR', TAU_CAT, 2 , graph_colors, 'strongly held beliefs and sensory', BEHAVIORAL_SAVE_DIR + 'strongly held beliefs and sensory_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
df_high_vs_low_shb[TAU_CAT].isna().sum(), df_high_vs_low_shb[STRONGLY_HELD_BELIEFS].isna().sum()

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_shb[df_high_vs_low_shb[TAU_CAT] == 'Low'][STRONGLY_HELD_BELIEFS]
group2 = df_high_vs_low_shb[df_high_vs_low_shb[TAU_CAT] == 'High'][STRONGLY_HELD_BELIEFS]

p_strongly_held_beliefs = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_shb
group2 = [TAU_CAT, 'SHB_THR']

p_strongly_held_beliefs_thr = stat_test(data_type, group1, group2)

### iADL

In [ ]:
IADL = 'iADL'

df_high_vs_low_iadl = df_high_vs_low.dropna(subset=[IADL])
df_high_vs_low_iadl[IADL] = df_high_vs_low_iadl[IADL].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_iadl, IADL, TAU_CAT, graph_colors, 'iADL',BEHAVIORAL_SAVE_DIR + 'iADL_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_iadl[df_high_vs_low_iadl[TAU_CAT] == 'Low'][IADL]
group2 = df_high_vs_low_iadl[df_high_vs_low_iadl[TAU_CAT] == 'High'][IADL]

p_iadl = stat_test(data_type, group1, group2)

# Statistical Correction
I'm using BH-FDR

In [ ]:
ALPHA = 0.05
METHOD = 'fdr_bh'

In [ ]:
# Statistical Correction
p_values = [p_trail_b, p_mbi_c, p_ist, p_intreset, p_impulse_control, p_following_societal_norms, p_strongly_held_beliefs, p_iadl, p_mbi_c_thr, p_intreset_thr, p_impulse_control_thr, p_following_societal_norms_thr, p_strongly_held_beliefs_thr]
risk_factors = ['TMT', 'MBI-C', 'IST', 'Interest, Motivation, and Drive', 'Impulse Control', 'Following Societal Norms', 'Strongly Held Beliefs', 'iADL', 'MBI-C Threshold', 'Interest, Motivation, and Drive Threshold', 'Impulse Control Threshold', 'Following Societal Norms Threshold', 'Strongly Held Beliefs Threshold']
p_values = [p for p in p_values if p is not None]
reject, p_values_corrected, _, _ = smm.multipletests(p_values, alpha=ALPHA, method=METHOD)

for i, p in enumerate(p_values):
    if p is not None:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

print("\nSignificant features:")
for i, p in enumerate(p_values):
    if p < ALPHA:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

In [ ]:
HIGH_TAU_217_THR

# Results
### 0.53
* No tests were significant

### 0.449
* No tests were significant